# Colab GPU Smoke Test

Verify the SiamAPN++ + MobileOne-S2 pipeline on a Colab GPU.

## Setup

In [ ]:
!git clone https://github.com/Fam-S/uav-person-tracker.git

In [ ]:
%cd uav-person-tracker

In [ ]:
!ls 

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!uv sync

In [ ]:
!mkdir -p checkpoints
!curl -L -o checkpoints/mobileone_s2_unfused.pth.tar https://docs-assets.developer.apple.com/ml-research/datasets/mobileone/mobileone_s2_unfused.pth.tar

## Verify CUDA

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## Shape Verification

In [ ]:
import torch

from models.backbone import MobileOneS2Backbone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = MobileOneS2Backbone(pretrained_path=None, normalize_input=True).to(device)

template = torch.rand(1, 3, 127, 127, device=device)
search = torch.rand(1, 3, 287, 287, device=device)

t_low, t_high = backbone(template)
s_low, s_high = backbone(search)

print("template 127x127 -> low:", tuple(t_low.shape), "high:", tuple(t_high.shape))
print("search   287x287 -> low:", tuple(s_low.shape), "high:", tuple(s_high.shape))

## Forward Pass

In [ ]:
import torch

from models import SiamAPNppMobileOne

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SiamAPNppMobileOne(feature_channels=96, pretrained_path=None).to(device)
template = torch.rand(1, 3, 127, 127, device=device)
search = torch.rand(1, 3, 255, 255, device=device)

outputs = model(template, search)
print("bbox_pred shape:", tuple(outputs["bbox_pred"].shape))
assert outputs["bbox_pred"].shape == (1, 4), "Expected (1, 4) bbox prediction"
print("Forward pass OK")

## Backward Pass & Loss

In [ ]:
import torch

from models import SiamAPNppMobileOne
from models.losses import SiamAPNLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SiamAPNppMobileOne(feature_channels=96, pretrained_path=None).to(device)
criterion = SiamAPNLoss(search_size=255).to(device)

template = torch.rand(1, 3, 127, 127, device=device)
search = torch.rand(1, 3, 255, 255, device=device)
search_bbox = torch.tensor([[60.0, 70.0, 50.0, 80.0]], device=device)

outputs = model(template, search)
loss_out = criterion(bbox_pred=outputs["bbox_pred"], search_bbox=search_bbox)

print("total_loss:", loss_out.total_loss.item())
print("reg_loss:", loss_out.reg_loss.item())
assert torch.isfinite(loss_out.total_loss), "Loss is not finite"

loss_out.total_loss.backward()
grads_exist = any(p.grad is not None for p in model.parameters() if p.requires_grad)
print("gradients flow:", grads_exist)
assert grads_exist, "No gradients flowing"
print("Backward pass OK")

## Architecture Tests

In [ ]:
!uv run pytest tests/test_architecture.py -v